# 02 - Analise temporal, receita e categorias

Aqui eu olho para volume de pedidos, receita, frete, estados e categorias. A base usada e o `olist_colab.sqlite` criado no notebook 01 ou baixado da Release.

In [ ]:
from pathlib import Path
import sqlite3
import urllib.request
import zipfile

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

PROJECT_DIR = Path.cwd()
DB_ZIP_URL = "https://github.com/Urpia-S/Olist_E-commerce_Analytic-SQL-Python/releases/download/data-v1/olist_colab.sqlite.zip"
OUTPUT_DIR = PROJECT_DIR / "outputs_colab"
DB_PATH = PROJECT_DIR / "olist_colab.sqlite"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def baixar_banco_da_release():
    zip_path = PROJECT_DIR / "olist_colab.sqlite.zip"
    urllib.request.urlretrieve(DB_ZIP_URL, zip_path)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(PROJECT_DIR)

    print("Banco extraido em:", DB_PATH)


if not DB_PATH.exists():
    baixar_banco_da_release()

conn = sqlite3.connect(DB_PATH)


def consulta(sql):
    return pd.read_sql_query(sql, conn)


def salvar_consulta(sql, arquivo):
    df = consulta(sql)
    destino = OUTPUT_DIR / arquivo
    df.to_csv(destino, index=False)
    print(f"Arquivo salvo: {destino}")
    return df


def grafico_barras(df, x, y, titulo, rotacao=0, top=None):
    dados = df.head(top) if top else df
    ax = dados.plot(kind="bar", x=x, y=y, legend=False, figsize=(10, 4))
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel(y)
    plt.xticks(rotation=rotacao, ha="right" if rotacao else "center")
    plt.tight_layout()
    plt.show()


consulta("""
-- Objetos disponiveis no banco preparado.
SELECT
    type AS tipo,
    name AS objeto
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name
LIMIT 20;
""")

## 1. Pedidos ao longo do tempo

Primeiro verifico a evolucao dos pedidos por mes, dia da semana e hora.

In [ ]:
pedidos_por_mes = salvar_consulta("""
-- Agrupo pedidos por mes para enxergar evolucao temporal.
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS mes_pedido,
    COUNT(*) AS pedidos,
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS pedidos_entregues,
    COUNT(DISTINCT customer_id) AS clientes
FROM core_orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY strftime('%Y-%m', order_purchase_timestamp)
ORDER BY mes_pedido;
""", "pedidos_por_mes.csv")

pedidos_por_mes.head()

In [ ]:
grafico_barras(pedidos_por_mes, "mes_pedido", "pedidos", "Pedidos por mes", rotacao=75)

In [ ]:
pedidos_por_dia_semana = salvar_consulta("""
-- Distribuicao dos pedidos por dia da semana.
SELECT
    CAST(strftime('%w', order_purchase_timestamp) AS INTEGER) AS numero_dia_sqlite,
    CASE CAST(strftime('%w', order_purchase_timestamp) AS INTEGER)
        WHEN 0 THEN 'domingo'
        WHEN 1 THEN 'segunda'
        WHEN 2 THEN 'terca'
        WHEN 3 THEN 'quarta'
        WHEN 4 THEN 'quinta'
        WHEN 5 THEN 'sexta'
        WHEN 6 THEN 'sabado'
    END AS dia_semana,
    COUNT(*) AS pedidos
FROM core_orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY numero_dia_sqlite, dia_semana
ORDER BY numero_dia_sqlite;
""", "pedidos_por_dia_semana.csv")

pedidos_por_hora = salvar_consulta("""
-- Distribuicao dos pedidos por hora do dia.
SELECT
    CAST(strftime('%H', order_purchase_timestamp) AS INTEGER) AS hora_do_dia,
    COUNT(*) AS pedidos
FROM core_orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY CAST(strftime('%H', order_purchase_timestamp) AS INTEGER)
ORDER BY hora_do_dia;
""", "pedidos_por_hora.csv")

display(pedidos_por_dia_semana)
display(pedidos_por_hora.head())

## 2. Receita, frete e categorias

Nesta parte comparo receita mensal, categorias e estados dos clientes.

In [ ]:
metricas_receita_mensal = salvar_consulta("""
-- Receita mensal com ticket medio e frete medio por item.
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS mes_pedido,
    COUNT(DISTINCT o.order_id) AS pedidos,
    COUNT(DISTINCT c.customer_unique_id) AS clientes_unicos,
    ROUND(SUM(oi.price), 2) AS receita_produtos,
    ROUND(SUM(oi.freight_value), 2) AS receita_frete,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS receita_total_com_frete,
    ROUND(SUM(oi.price) / NULLIF(COUNT(DISTINCT o.order_id), 0), 2) AS ticket_medio_produtos,
    ROUND(AVG(oi.freight_value), 2) AS frete_medio_item
FROM core_orders o
JOIN core_customers c ON c.customer_id = o.customer_id
JOIN core_order_items oi ON oi.order_id = o.order_id
WHERE o.order_purchase_timestamp IS NOT NULL
GROUP BY strftime('%Y-%m', o.order_purchase_timestamp)
ORDER BY mes_pedido;
""", "metricas_receita_mensal.csv")

receita_por_categoria = salvar_consulta("""
-- Receita e frete por categoria traduzida.
SELECT
    categoria_ingles,
    COUNT(DISTINCT order_id) AS pedidos,
    COUNT(*) AS itens_vendidos,
    ROUND(SUM(price), 2) AS receita_produtos,
    ROUND(SUM(freight_value), 2) AS receita_frete,
    ROUND(SUM(price) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS ticket_medio_pedido_categoria,
    ROUND(AVG(freight_value), 2) AS frete_medio_item
FROM vw_order_items_enriched
GROUP BY categoria_ingles
ORDER BY receita_produtos DESC;
""", "receita_por_categoria.csv")

receita_por_categoria.head(10)

In [ ]:
grafico_barras(receita_por_categoria, "categoria_ingles", "receita_produtos", "Top categorias por receita", rotacao=70, top=10)

In [ ]:
receita_por_estado = salvar_consulta("""
-- Receita por estado do cliente.
SELECT
    c.customer_state AS estado_cliente,
    COUNT(DISTINCT o.order_id) AS pedidos,
    COUNT(DISTINCT c.customer_unique_id) AS clientes_unicos,
    ROUND(SUM(oi.price), 2) AS receita_produtos,
    ROUND(SUM(oi.freight_value), 2) AS receita_frete,
    ROUND(SUM(oi.price) / NULLIF(COUNT(DISTINCT o.order_id), 0), 2) AS ticket_medio_produtos
FROM core_orders o
JOIN core_customers c ON c.customer_id = o.customer_id
JOIN core_order_items oi ON oi.order_id = o.order_id
GROUP BY c.customer_state
ORDER BY receita_produtos DESC;
""", "receita_por_estado.csv")

receita_por_estado.head()